# The Screening Room demo

This notebook demonstrates a few read-only checks against the local Screening Room API.

Requirements:
- FastAPI app already running at http://localhost:8000
- Neo4j already running and populated with MovieLens data
- Ollama already running with the qwen2.5:1.5b model available

No passwords, tokens, or personal credentials are stored here.

In [ ]:
import json
import urllib.request

API_BASE = 'http://localhost:8000'

def get_json(path: str):
    with urllib.request.urlopen(f'{API_BASE}{path}') as response:
        return json.loads(response.read().decode('utf-8'))

print('API base:', API_BASE)
print('Health:', get_json('/health'))


## 1) Graph node and relationship checks

These examples assume the MovieLens graph has been imported and the database is available.
The recommendation and analytics endpoints provide the graph-level reading patterns we use to confirm app health and data availability.

In [ ]:
# Confirm the homepage is served and a recommendation endpoint responds.
with urllib.request.urlopen(f'{API_BASE}/') as response:
    html = response.read(200).decode('utf-8', errors='replace')
    print('Homepage served successfully:', html[:80].replace('\n', ' '))

popular = get_json('/recommendations/popular?limit=3')
print('Popular movies sample:')
print(json.dumps(popular, indent=2)[:600])


## 2) Popular, hidden-gem, and personalized recommendations

These calls hit the real endpoints defined in app/main.py:
- GET /recommendations/popular
- GET /recommendations/hidden-gems
- GET /recommendations/personalized/{user_id}

The payload structure is a {limit, items} response for the popular and hidden-gem routes, and a similar payload for personalized recommendations.

In [ ]:
popular = get_json('/recommendations/popular?limit=5')
hidden = get_json('/recommendations/hidden-gems?limit=5')
personalized = get_json('/recommendations/personalized/1?limit=5')

print('Popular:')
print(json.dumps(popular, indent=2))
print('Hidden gems:')
print(json.dumps(hidden, indent=2))
print('Personalized for user 1:')
print(json.dumps(personalized, indent=2))


## 3) GDS analytics: similar users and communities

These endpoints are implemented as:
- GET /analytics/similar-users/{user_id}
- GET /analytics/communities

The response shape is a top-level object including a user_id or limit and an items list.

In [ ]:
similar_users = get_json('/analytics/similar-users/1?limit=5')
communities = get_json('/analytics/communities?limit=5')

print('Similar users:')
print(json.dumps(similar_users, indent=2))
print('Communities:')
print(json.dumps(communities, indent=2))


## 4) Shortest-path checks

The path endpoints are:
- GET /analytics/path/users/{user_id_1}/{user_id_2}
- GET /analytics/path/user-movie/{user_id}/{movie_id}

These return a dict containing found, path, and message fields when no path exists.

In [ ]:
user_user_path = get_json('/analytics/path/users/1/2')
user_movie_path = get_json('/analytics/path/user-movie/1/1')

print('User-user path:')
print(json.dumps(user_user_path, indent=2))
print('User-movie path:')
print(json.dumps(user_movie_path, indent=2))


## 5) Ollama assistant POST /assistant/ask

The assistant endpoint is implemented as POST /assistant/ask and accepts a JSON object with a question string. It validates input, invokes the intent classifier through Ollama, and executes only allowlisted read-only queries.

In [ ]:
payload = json.dumps({'question': 'Find five highly rated comedy movies'}).encode('utf-8')
request = urllib.request.Request(
    f'{API_BASE}/assistant/ask',
    data=payload,
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urllib.request.urlopen(request) as response:
    assistant_result = json.loads(response.read().decode('utf-8'))
print(json.dumps(assistant_result, indent=2))


## Notes

- This notebook does not store credentials or personal data.
- The API base is intentionally set to http://localhost:8000.
- If the app, Neo4j, or Ollama is not already running, these requests will fail until those services are started.
- No outputs are fabricated; the cells are left ready for live execution in a running environment.